# TweetAge · 05 Experimentos v3: mejor suavizado (Kneser-Ney) y NB binarizado

El análisis de errores (03) mostró que los canales de caracteres son los que más votan y que el problema de fondo es la **dispersión**: muchos n-gramas del tweet nunca se vieron en train. Todos los canales usaban el suavizado más simple (sumar α a cada conteo). Aquí se prueban dos mejoras clásicas de los modelos de lenguaje de n-gramas, sin salir de las reglas (siguen siendo LMs propios de n-gramas, entrenados solo con los datos del reto).

### En palabras simples

- **Kneser-Ney (KN).** Un LM de caracteres estima la probabilidad de la siguiente letra dado lo que viene antes (por ejemplo, después de `jaj` viene `a`). Si un contexto nunca se vio, hay que "inventar" una probabilidad. El suavizado add-α le da a todo lo no visto la misma probabilidad pequeña. KN es más inteligente: le quita un poco de probabilidad a lo que sí se vio y la reparte según un modelo con **menos contexto** (si no conozco `xjaj`, miro qué pasa después de `jaj`, y si no, después de `aj`...). Además, en los modelos de menos contexto cuenta **en cuántos contextos distintos** aparece cada n-grama, no cuántas veces: así una letra que solo aparece en una palabra muy repetida no recibe crédito de más. Es el suavizado estándar de los LMs de n-gramas (Chen y Goodman, 1998).
- **NB binarizado.** El Naive Bayes de caracteres cuenta cuántas veces aparece cada n-grama en el tweet. El binarizado cuenta solo si aparece o no (una vez por tweet), y junta varios órdenes (3–7) en un solo modelo. En textos cortos suele ser más robusto.

### Protocolo (el mismo del 02)

Se parte de `greedy_ent` (0.5937 en confirmación, 0.60796 en Kaggle). Cada movimiento (agregar un canal, cambiar un canal agregado o reemplazar un canal de la mezcla) elige su temperatura y peso **solo en la semilla de ajuste**. En cada ronda se prueba el mejor movimiento de ajuste y se acepta solo si también mejora en la de confirmación. Al final se compara con bootstrap.

Código: `models_kn.py` (modelos), `channels_extra.py` (canales `kn_*` y `nbbin_*`), `experiments_v3.py` (búsqueda y alternativas). La spec se guarda como `greedy_kn` para usarla en `04_submission_v2.ipynb`.

In [ ]:
import numpy as np
import pandas as pd

from tweetage import channels_extra as CX
from tweetage import error_analysis as EA
from tweetage import evaluation as E
from tweetage import experiments_v2 as XV
from tweetage import experiments_v3 as X3
from tweetage import paths
from tweetage.pipeline import CLASES

N_JOBS = 6  # los modelos KN de orden 7 usan bastante memoria; con 1 proceso se entrenan uno a uno
AJ, CF = EA.SEED_AJUSTE, EA.SEED_CONFIRMACION
NUEVOS = CX.KN_NAMES + CX.NBBIN_NAMES
pd.set_option("display.max_colwidth", 120)

train, _ = paths.load_data()
X = paths.texts(train)
y = train["age_range"].to_numpy()

M = {}
for s in (AJ, CF):
    m = EA.cargar(X, y, s, "greedy_ent", 2)  # canales de greedy_ent (caché del 02)
    m.probs_base.update(
        XV.run_seed(X, y, s, CX.VARIANTS["base"], NUEVOS, N_JOBS).probs
    )  # nuevos (~6 min por semilla la 1.ª vez)
    M[s] = m
EA.verificar(M).round(4)

## 1. Cada canal por sí solo

F1 macro de cada canal nuevo comparado con el mejor canal equivalente que ya existía. `kn_char*_cs` = LM de caracteres con Kneser-Ney (orden = tamaño del n-grama, `_cs` = respeta mayúsculas); `kn_word*` = LM de palabras con KN; `nbbin_*` = NB binarizado con los órdenes indicados.

In [ ]:
nombres = NUEVOS + ["char5_cs", "char6_cs", "char_seq4", "word_seq3_ent5_tt"]
solo = pd.DataFrame(
    {
        n: {
            "ajuste": E.f1_macro(M[AJ].y_idx, M[AJ].probs_base[n]),
            "confirmación": E.f1_macro(M[CF].y_idx, M[CF].probs_base[n]),
        }
        for n in nombres
    }
).T
solo["tipo"] = ["nuevo"] * len(NUEVOS) + ["anterior"] * 4
solo.sort_values("confirmación", ascending=False).round(4)

**Qué vemos.** Los LM de caracteres con Kneser-Ney son los mejores canales individuales del proyecto (≈ 0.565, frente a 0.553 del mejor anterior), y el LM de palabras con KN sube de 0.461 a 0.482. El NB binarizado también mejora al NB normal (0.561 con órdenes 4–7 frente a 0.553), pero queda por debajo de KN.

## 2. Búsqueda greedy sobre `greedy_ent`

Movimientos que se prueban:

- **agregar** un canal KN o NB binarizado encima de la mezcla;
- **cambiar** un canal agregado (`shape7`, `char_seq4`) por uno nuevo;
- **reemplazar** `char4`, `char5`, `char6` o el LM de palabras por un canal nuevo, en la misma posición y con el mismo peso (con su propia temperatura).

Tarda ≈ 2 minutos.

In [ ]:
ADD = [
    "kn_char5_cs",
    "kn_char6_cs",
    "kn_char7_cs",
    "kn_char6",
    "nbbin_c456_cs",
    "nbbin_c3456_cs",
    "nbbin_c4567_cs",
    "kn_word3_ent5_tt",
]
CHAR = [
    "kn_char5_cs",
    "kn_char6_cs",
    "kn_char7_cs",
    "nbbin_c5_cs",
    "nbbin_c456_cs",
    "nbbin_c3456_cs",
    "nbbin_c4567_cs",
]
SWAP = {"word_seq3": ["kn_word3_ent5_tt", "kn_word4_ent5_tt"], "char4": CHAR, "char5": CHAR, "char6": CHAR}

spec_kn, rondas = X3.greedy(M[AJ].spec, M[AJ], M[CF], ADD, SWAP)
rondas.round(4)

In [ ]:
print("Reemplazos:", spec_kn["swaps"], "| temperaturas:", spec_kn.get("swap_temps"))
pd.DataFrame(spec_kn["adds"])

## 3. Resultado frente a `greedy_ent` y a la mezcla vigente

In [ ]:
p_kn = {s: M[s].predict(spec_kn) for s in M}
filas = [
    E.compare(
        "greedy_kn vs greedy_ent",
        M[AJ].y_idx,
        M[AJ].p_greedy,
        p_kn[AJ],
        M[CF].y_idx,
        M[CF].p_greedy,
        p_kn[CF],
    ),
    E.compare("greedy_kn vs vigente", M[AJ].y_idx, M[AJ].p_vig, p_kn[AJ], M[CF].y_idx, M[CF].p_vig, p_kn[CF]),
]
cols = [
    "f1_ajuste",
    "f1_confirmacion",
    "delta_ajuste",
    "delta_confirmacion",
    "ic95_bajo",
    "ic95_alto",
    "pred_cambiadas",
    "pasa",
    "fuerte",
]
pd.DataFrame(filas).set_index("experimento")[cols].round(4)

In [ ]:
display(
    pd.DataFrame(
        {
            "vigente": E.f1_per_class(M[CF].y_idx, M[CF].p_vig),
            "greedy_ent": E.f1_per_class(M[CF].y_idx, M[CF].p_greedy),
            "greedy_kn": E.f1_per_class(M[CF].y_idx, p_kn[CF]),
        }
    ).T.round(3)
)
pd.DataFrame(
    {
        "real": E.pred_distribution(np.eye(6)[M[CF].y_idx]),
        "greedy_ent": E.pred_distribution(M[CF].p_greedy),
        "greedy_kn": E.pred_distribution(p_kn[CF]),
    }
).T

In [ ]:
fila = filas[0]
spec_kn.update(
    {
        "experimento": "v3_greedy_kn",
        "f1_confirmacion": float(fila["f1_confirmacion"]),
        "delta_confirmacion": float(filas[1]["delta_confirmacion"]),
    }
)
print(XV.save_spec("greedy_kn", spec_kn))

**Qué vemos.** La búsqueda aceptó 4 movimientos, todos con Kneser-Ney: `char6` se reemplaza por `kn_char6_cs`, el `char_seq4` agregado se cambia por `kn_char7_cs` y se agrega `kn_char6` (con casefold). Frente a `greedy_ent` mejora +0.0044 en ajuste y +0.0054 en confirmación, con intervalo al 95 % de [+0.0031, +0.0078], y sube el F1 de las 6 clases. Frente a la mezcla vigente son +0.0123 en confirmación. Con la relación local → Kaggle observada (≈ 1:1), se espera ≈ 0.613 en el leaderboard público.

Ni el NB binarizado ni el LM de palabras con KN entraron: mejoran por sí solos, pero lo que aportan ya lo cubren los canales KN de caracteres y el LM de entidades.

**Advertencia:** igual que en el 02, aceptar cada paso miró las dos semillas, así que el Δ de confirmación es algo optimista.

## 4. Alternativas probadas que no se adoptan

### 4.1 Mezcla log-lineal con pesos optimizados para F1

En lugar de promediar probabilidades, se suman log-probabilidades con un peso por canal (Σ wᵢ·log pᵢ) y los pesos se buscan juntos por coordenadas, maximizando el F1 de ajuste.

In [ ]:
tabla_ll, pesos_ll = X3.loglineal_f1(spec_kn, M[AJ], M[CF])
display(tabla_ll.round(4))
print(
    f"Mezcla lineal (greedy_kn): ajuste {E.f1_macro(M[AJ].y_idx, p_kn[AJ]):.4f} | confirmación {E.f1_macro(M[CF].y_idx, p_kn[CF]):.4f}"
)
pesos_ll.round(3).to_frame("peso")

**Qué vemos.** La log-lineal queda por debajo de la mezcla lineal en ambas semillas (0.5999 / 0.5977 frente a 0.6009 / 0.5992). El producto de expertos le da la última palabra al canal más seguro de sí mismo, y los NB de caracteres son muy "seguros" aunque se equivoquen; el promedio lineal amortigua esos excesos. Coincide con lo visto en el 02 (B4).

### 4.2 Sesgos por clase sobre la nueva mezcla

Se ajustan 6 sesgos en una semilla y se evalúan en la otra, en los dos sentidos.

In [ ]:
X3.sesgos_cruzados(p_kn[AJ], M[AJ].y_idx, p_kn[CF], M[CF].y_idx).round(4)

**Qué vemos.** Ajustados en la semilla de ajuste mejoran la de confirmación (+0.0016), pero ajustados en la de confirmación empeoran la de ajuste (−0.0008), y ningún intervalo excluye el 0. Los sesgos óptimos cambian de una partición a otra (sobreajuste), así que no se adoptan.

### 4.3 log(1 + tf) y Complement NB

`log(1+tf)` por tweet es prácticamente el NB binarizado: en tweets de ≤ 140 caracteres casi todos los n-gramas aparecen una sola vez. Complement NB corrige sobre todo el sesgo hacia clases grandes, y aquí las clases están balanceadas (16–17 % cada una), así que no se esperaba ganancia y no se implementó.

## 5. Conclusión

Para un LM de n-gramas, el suavizado importa más que agregar canales: cambiar add-α por Kneser-Ney en los canales de caracteres da la mayor mejora individual de la tanda. La spec `greedy_kn` es la candidata para Kaggle: en `04_submission_v2.ipynb`, poner `SPEC = "greedy_kn"` y `N_JOBS = 1` o `2`.